Importing libraries 

In [12]:
import torch 
import numpy
import tqdm
import pandas 



Dataset download - wikitext2 

In [13]:
from datasets import load_dataset

ds = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    cache_dir="E:/projects/llm/llm/data/"
)

train_text = " ".join(ds["train"]["text"])
val_text   = " ".join(ds["validation"]["text"])
test_text  = " ".join(ds["test"]["text"])


Custom tokenizer  - character level 

In [14]:

text = train_text 
#Making a list of unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

stringtoindex = {ch:i for i,ch in enumerate(chars)}
indextostring= {i:ch for i, ch in enumerate(chars)}


#print("Here is stringtoindex \n", stringtoindex )
#print("Here is indextostring \n", indextostring)


#Encoding

data=torch.tensor([stringtoindex[c] for c in text], dtype=torch.long)
print(data)

tensor([ 1,  1, 30,  ...,  1,  0,  1])


Training the model

In [15]:
#Creating batches to train 

def get_batch(data, batch_size, seq_len):

    # randint inputs - lower bound, upper bound, and dimension 
    ix = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    
    return x, y



In [16]:
#Setting the dimensions of the model 

import torch.nn as nn 
import torch.nn.functional as F 
import math 


#Self attention class 

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        att = (Q @ K.transpose(-2, -1)) / math.sqrt(x.size(-1))
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)

        return att @ V


# Transformer structure and layers 

class Block(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = SelfAttention(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

Model stucture and forward pass

In [17]:
class Model(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, seq_len=128):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)

        self.blocks = nn.ModuleList([
            Block(d_model) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

        self.seq_len = seq_len

    def forward(self, x, targets=None):
        B, T = x.shape

        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        x = tok + pos

        mask = torch.tril(torch.ones(T, T, device=x.device))

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        logits = self.head(x)

        if targets is None:
            return logits

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.view(-1)
        )

        return logits, loss

Training Loop

In [25]:
from tqdm import tqdm 

#Set default device to gpu
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

model = Model(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

pbar = tqdm(range(10000))
for step in pbar:
    x, y = get_batch(data, 32, 128)
    x=x.to(device)
    y=y.to(device)

    logits, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    pbar.set_description(f"loss: {loss.item():.4f}")

cuda:0


loss: 1.2905: 100%|██████████| 10000/10000 [07:11<00:00, 23.18it/s]


Generate tokens funciton

In [31]:
def generate(model, start, stoi, itos, max_new_tokens=200, temperature=1.0, top_k=None, device="cuda:0"):
    model.eval()
    
    x = torch.tensor([stoi[c] for c in start], dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -model.seq_len:]  # crop if too long
        
        logits = model(x_cond)
        logits = logits[:, -1, :] / temperature
        
        probs = F.softmax(logits, dim=-1)

        if top_k is not None:
            v, ix = torch.topk(probs, top_k)
            probs = torch.zeros_like(probs).scatter_(1, ix, v)
            probs = probs / probs.sum()

        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token], dim=1)

    return "".join([itos[i] for i in x[0].tolist()])

Sanity checks + testing model 

In [34]:
def compute_ppl(loss):
    return math.exp(loss)

model.eval()

x, y = get_batch(data, 32, 128)
x, y = x.to(device), y.to(device)

with torch.no_grad():
    _, loss = model(x, y)

print("loss:", loss.item())
print("ppl:", compute_ppl(loss.item()))

print(generate(model, "The ", stringtoindex, indextostring, temperature=0.8, top_k=50, device=device))


loss: 1.268662929534912
ppl: 3.5560946331160284
The invisible place on the crickets , Braathens , which was on a flow standard religious supported the commanded the claim . However , Tete KAP 's officer claimed and Street , Continental Struct , and Oly
